In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração para mostrar números sem notação científica (ex: 1,000,000.00)
pd.options.display.float_format = '{:,.2f}'.format

# --- FUNÇÕES DE LIMPEZA ---
def limpar_valor(valor):
    """Transforma '124M €' em 124000000.0"""
    if pd.isna(valor): return 0
    if isinstance(valor, (int, float)): return valor
    valor = str(valor).replace('€', '').strip()
    if 'M' in valor: return float(valor.replace('M', '')) * 1_000_000
    if 'K' in valor: return float(valor.replace('K', '')) * 1_000
    return float(valor)

def limpar_idade(valor):
    """Transforma '7 fev 1999 (26)' em 26"""
    if pd.isna(valor): return 0
    valor = str(valor)
    if '(' in valor:
        return int(valor.split('(')[-1].replace(')', ''))
    return 0

print("Bibliotecas e funções carregadas!")

Bibliotecas e funções carregadas!


In [ ]:
# 1. Carregar Jogadores (Verifique o nome do arquivo)
df = pd.read_csv('jogadores/sofascore_premier_league_geral_final_valor_de_mercado_24_25.csv')
df['Liga'] = 'Premier League'

# 2. Aplicar Limpeza
df['Valor_Numerico'] = df['Valor de mercado'].apply(limpar_valor)
df['Idade_Limpa'] = df['Idade'].apply(limpar_idade)
df['Passe_Numerico'] = df['Acerto no passe %'].apply(lambda x: float(str(x).replace('%','')) if pd.notna(x) else 0)

# 3. CORREÇÃO DOS JOGADORES (O "Patch" Bundesliga)
# Mapeia jogadores para os times corretos da temporada 24/25
correcao_jogadores = {
    # --- PREMIER LEAGUE ---
    'Julián Álvarez': 'Manchester City',
    'João Cancelo': 'Manchester City', # Ou Barcelona se considerar o empréstimo
    'Ivan Toney': 'Brentford',
    'Douglas Luiz': 'Aston Villa',
    'Moussa Diaby': 'Aston Villa',
    'Dominic Solanke': 'Bournemouth',
    'Pedro Neto': 'Wolverhampton',
    'Max Kilman': 'Wolverhampton',
    'Michael Olise': 'Crystal Palace',
    'Joachim Andersen': 'Crystal Palace',
    'João Palhinha': 'Fulham',
    'Tosin Adarabioyo': 'Fulham',
    'Amadou Onana': 'Everton',
    'Conor Gallagher': 'Chelsea',
    'Thiago Silva': 'Chelsea',
    'Ian Maatsen': 'Chelsea', # Ou Dortmund
    'Emile Smith Rowe': 'Arsenal',
    'Eddie Nketiah': 'Arsenal',
    'Aaron Ramsdale': 'Arsenal',
    'Scott McTominay': 'Manchester United',
    'Aaron Wan-Bissaka': 'Manchester United',
    'Mason Greenwood': 'Getafe',
}


df['Time_Original'] = df['Time']
df['Time'] = df['Nome'].map(correcao_jogadores).fillna(df['Time'])

print(f"Jogadores carregados: {len(df)}")


Jogadores carregados: 317


In [ ]:
# 1. Carregar Tabela
df_tabela = pd.read_csv('tabela/tabela_premier_league_24_25.csv')

df_tabela.columns = df_tabela.columns.str.strip()

mapa_colunas = {'Posição': 'Pos', '#': 'Pos', 'Rank': 'Pos', 'RK': 'Pos'}
df_tabela.rename(columns=mapa_colunas, inplace=True)

# 4. Tratar Gols (XX:YY) para Gols Pró e Contra
if 'Gols' in df_tabela.columns:
    gols_split = df_tabela['Gols'].str.split(':', expand=True)
    if len(gols_split.columns) == 2:
        df_tabela['Gols_Pro'] = pd.to_numeric(gols_split[0])
        df_tabela['Gols_Contra'] = pd.to_numeric(gols_split[1])

mapa_times_tabela = {
    # Manchester City
    'Man City': 'Manchester City',
    'Manchester City': 'Manchester City',
    
    # Manchester United
    'Man Utd': 'Manchester United',
    'Manchester Utd': 'Manchester United',
    'Manchester United': 'Manchester United',
    
    # Tottenham
    'Spurs': 'Tottenham Hotspur',
    'Tottenham': 'Tottenham Hotspur',
    'Tottenham Hotspur': 'Tottenham Hotspur',
    
    # Wolverhampton
    'Wolves': 'Wolverhampton',
    'Wolverhampton': 'Wolverhampton',
    'Wolverhampton Wanderers': 'Wolverhampton',
    
    # Brighton
    'Brighton': 'Brighton & Hove Albion',
    'Brighton & Hove Albion': 'Brighton & Hove Albion',
    
    # West Ham
    'West Ham': 'West Ham United',
    'West Ham United': 'West Ham United',
    
    # Newcastle
    'Newcastle': 'Newcastle United',
    'Newcastle United': 'Newcastle United',
    
    # Nottingham Forest
    'Nottm Forest': 'Nottingham Forest',
    'Nottingham': 'Nottingham Forest',
    'Nottingham Forest': 'Nottingham Forest',
    'Nott\'m Forest': 'Nottingham Forest',
    
    # Leicester (Subiu)
    'Leicester': 'Leicester City',
    'Leicester City': 'Leicester City',
    
    # Ipswich (Subiu)
    'Ipswich': 'Ipswich Town',
    'Ipswich Town': 'Ipswich Town',
    
    # Southampton (Subiu)
    'Southampton': 'Southampton',
    
    # Outros times (Geralmente iguais, mas por garantia)
    'Arsenal': 'Arsenal',
    'Aston Villa': 'Aston Villa',
    'Bournemouth': 'Bournemouth', # Às vezes aparece como AFC Bournemouth
    'AFC Bournemouth': 'Bournemouth',
    'Brentford': 'Brentford',
    'Chelsea': 'Chelsea',
    'Crystal Palace': 'Crystal Palace',
    'Everton': 'Everton',
    'Fulham': 'Fulham',
    'Liverpool': 'Liverpool',
    
    # Caso você tenha dados antigos de times rebaixados (Segurança)
    'Luton': 'Luton Town',
    'Burnley': 'Burnley',
    'Sheffield Utd': 'Sheffield United'
}
df_tabela['Time'] = df_tabela['Time'].str.strip().replace(mapa_times_tabela)

elenco_stats = df.groupby('Time').agg({
    'Valor_Numerico': 'sum',
    'Média das notas Sofascore': 'mean'
}).reset_index().rename(columns={'Valor_Numerico': 'Valor_Total_Elenco', 'Média das notas Sofascore': 'Media_Sofa_Time'})

# Merge Tabela + Elenco
df_times_final = pd.merge(df_tabela, elenco_stats, on='Time', how='inner')

# Merge Jogadores + Tabela Completa
df_final = pd.merge(df, df_times_final[['Time', 'Pos', 'PTS', 'Media_Sofa_Time', 'Valor_Total_Elenco']], on='Time', how='inner')

print(f"Merge concluído! Total de jogadores com dados completos: {len(df_final)}")

Merge concluído! Total de jogadores com dados completos: 238


In [9]:
# 1. CÁLCULO DO ÍNDICE ATLAS (Performance Individual vs Coletiva)
df_final['Indice_Atlas'] = df_final['Média das notas Sofascore'] - df_final['Media_Sofa_Time']

print("=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===")
display(df_final.sort_values('Indice_Atlas', ascending=False)[['Nome', 'Time', 'Pos', 'Média das notas Sofascore', 'Indice_Atlas']].head(10))

# 2. ANÁLISE DE CONTEXTO (Luxo na Lama)
print("\n=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===")
# Critério: Valor > 20M e Posição > 9
luxo = df_final[
    (df_final['Valor_Numerico'] > 20_000_000) & 
    (df_final['Pos'] > 9)
].sort_values('Valor_Numerico', ascending=False)

if len(luxo) > 0:
    display(luxo[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])
else:
    print("Nenhum jogador encontrado com esses critérios na Bundesliga.")

# 3. ANÁLISE DE CONTEXTO (Diamantes na Lama)
print("\n=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===")
# Critério: Nota > 7.15 e Posição > 9
diamantes = df_final[
    (df_final['Média das notas Sofascore'] > 7.15) & 
    (df_final['Pos'] > 9)
].sort_values('Média das notas Sofascore', ascending=False)

display(diamantes[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])

=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===


,Nome,Time,Pos,Média das notas Sofascore,Indice_Atlas
0,Mohamed Salah,Liverpool,1,7.78,0.68
14,Cole Palmer,Chelsea,4,7.63,0.60
28,Bruno Fernandes,Manchester United,15,7.62,0.54
29,Matheus Cunha,Manchester United,15,7.61,0.53
43,Bukayo Saka,Arsenal,2,7.60,0.51
63,Youri Tielemans,Aston Villa,6,7.38,0.42
30,Bryan Mbeumo,Manchester United,15,7.47,0.39
87,James Maddison,Tottenham Hotspur,17,7.30,0.39
31,Amad Diallo,Manchester United,15,7.42,0.34
111,Alex Iwobi,Fulham,11,7.27,0.32



=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
29,Matheus Cunha,Manchester United,15,73M €,7.61
30,Bryan Mbeumo,Manchester United,15,68M €,7.47
174,Adam Wharton,Crystal Palace,12,63M €,6.97
95,Mohammed Kudus,Tottenham Hotspur,17,60M €,6.82
197,Pedro Neto,Wolverhampton,16,53M €,7.01
...,...,...,...,...,...
175,Tyrick Mitchell,Crystal Palace,12,23M €,6.94
34,Aaron Wan-Bissaka,Manchester United,15,23M €,7.13
220,Leif Davis,Ipswich Town,19,23M €,6.77
117,Tosin Adarabioyo,Fulham,11,21M €,6.98



=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
28,Bruno Fernandes,Manchester United,15,41M €,7.62
29,Matheus Cunha,Manchester United,15,73M €,7.61
30,Bryan Mbeumo,Manchester United,15,68M €,7.47
31,Amad Diallo,Manchester United,15,39M €,7.42
87,James Maddison,Tottenham Hotspur,17,34M €,7.30
99,James Tarkowski,Everton,13,NaN,7.30
111,Alex Iwobi,Fulham,11,30M €,7.27
100,Jordan Pickford,Everton,13,19.3M €,7.24
137,Mikkel Damsgaard,Brentford,10,29M €,7.20
149,Jarrod Bowen,West Ham United,14,39M €,7.20


In [ ]:
def ficha_jogador(parte_do_nome):
    """
    Busca um jogador pelo nome (ou parte dele) e mostra todas as colunas.
    """
    dataframe_alvo = df 
    
    # 2. Faz a busca (converte tudo para minúsculo para não ter erro de maiúscula/minúscula)
    mask = dataframe_alvo['Nome'].str.lower().str.contains(parte_do_nome.lower(), na=False)
    resultado = dataframe_alvo[mask]
    
    # 3. Verifica se achou
    if len(resultado) == 0:
        return f"❌ Nenhum jogador encontrado com o nome '{parte_do_nome}'."

    return resultado.T


nome_para_buscar = input("Digite o nome (ou parte do nome) do jogador: ")

display(ficha_jogador(nome_para_buscar))

,46
Link Jogador,https://www.sofascore.com/pt/football/player/c...
Time,Manchester United
Valor de mercado,9.1M €
Idade,23 de fevereiro de 1992 (33)
Pontos fortes,Posicionamento; Desarme; Duelos no chão
Pontos fracos,Disciplina
Posição,Midfielder
Gols,1
Gols esperados (xG),1.83
Dribles certos,3


In [ ]:
# Preparando dados para o gráfico
grafico_df = df_times_final.copy()

# Tratamento para evitar erro no gráfico (Divisão por zero ou NaN)
grafico_df['PTS'] = pd.to_numeric(grafico_df['PTS'], errors='coerce').fillna(0)
grafico_df['Valor_Total_Elenco'] = pd.to_numeric(grafico_df['Valor_Total_Elenco'], errors='coerce').fillna(0)
pts_ajustado = grafico_df['PTS'].replace(0, 1) # Evita dividir por zero

grafico_df['Custo_Por_Ponto'] = grafico_df['Valor_Total_Elenco'] / pts_ajustado

fig = px.scatter(grafico_df, 
                 x='Valor_Total_Elenco', 
                 y='PTS', 
                 text='Time', 
                 size='Custo_Por_Ponto', 
                 color='Pos',
                 color_continuous_scale='Bluered_r', # Azul = Topo, Vermelho = Baixo
                 hover_data=['Custo_Por_Ponto'],
                 title='Eficiência Financeira Bundesliga: Valor do Elenco x Pontos',
                 labels={'Valor_Total_Elenco': 'Investimento Total (€)', 'PTS': 'Pontos Conquistados'})

fig.update_traces(textposition='top center')
fig.update_yaxes(range=[0, 100], autorange=False, dtick=10)
fig.show()

In [ ]:
def scouting_premierleague(posicao=None, orcamento_max=None, atributo_obrigatorio=None):
    res = df_final.copy()
    
    if posicao:
        res = res[res['Posição'].str.contains(posicao, case=False, na=False)]
    
    if orcamento_max:
        valor = float(orcamento_max) * 1_000_000
        res = res[res['Valor_Numerico'] <= valor]
        
    if atributo_obrigatorio:
        res = res[res['Pontos fortes'].str.contains(atributo_obrigatorio, case=False, na=False)]
        
    cols = ['Nome', 'Time', 'Idade_Limpa', 'Valor de mercado', 'Média das notas Sofascore', 'Pontos fortes']
    return res.sort_values('Média das notas Sofascore', ascending=False)[cols].head(10)


print("🔍 Resultado Scouting:")
display(scouting_premierleague(posicao='Midfielder', orcamento_max=50))

🔍 Resultado Scouting (Meias, Max 40M, Bom de Passe):


,Nome,Time,Idade_Limpa,Valor de mercado,Média das notas Sofascore,Pontos fortes
0,Mohamed Salah,Liverpool,33,44M €,7.78,Alta pressão; Armador; Marcador de pênalti
28,Bruno Fernandes,Manchester United,31,41M €,7.62,Armador; Finalizações de longe; Cobranças dire...
63,Youri Tielemans,Aston Villa,28,41M €,7.38,Passes longos; Armador; Passe
77,Mateo Kovačić,Manchester City,31,19.4M €,7.30,Consistência; Armador; Duelos no chão
87,James Maddison,Tottenham Hotspur,28,34M €,7.30,Alta pressão; Armador; Passe
111,Alex Iwobi,Fulham,29,30M €,7.27,Armador; Alta pressão
81,Bernardo Silva,Manchester City,31,31M €,7.23,Alta pressão; Armador; Passe
137,Mikkel Damsgaard,Brentford,25,29M €,7.20,Armador; Alta pressão
149,Jarrod Bowen,West Ham United,28,39M €,7.20,Armador; Alta pressão
32,Casemiro,Manchester United,33,9.1M €,7.18,Posicionamento; Desarme; Duelos no chão
